## Гетерогенные эффекты: мета-обучатели, причинный лес и проверка оценки CATE

**Материал подготовлен для курса [Объяснимое машинное обучение](https://ai-interpretability.school), модуль «Гетерогенные эффекты».**

В модуле мы разобрали, почему индивидуальный эффект $\tau_i = Y_i(1) - Y_i(0)$ не наблюдается ни у кого, как условный средний эффект $\tau(x)$ оценивают мета-обучатели и причинный лес и почему проверить оценку можно только на группах объектов. В этой тетради все это проделывается на синтетических данных, где истинный $\tau(x)$ известен:

1. S-, T- и X-learner на EconML и рассеяние «истинный эффект против оцененного»;
2. причинный лес `CausalForestDML` с доверительными интервалами;
3. калибровка по квантилям предсказанного эффекта;
4. uplift-кривая, кривая Qini и коэффициент Qini.

На реальных данных сравнение с истинным эффектом невозможно, поэтому шаги 3 и 4 — те проверки, которые остаются в работе с настоящими данными, а шаги 1 и 2 показывают, насколько они согласуются с истиной.

In [ ]:
!pip install -q econml==0.17.0

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from econml.metalearners import SLearner, TLearner, XLearner
from econml.dml import CausalForestDML

### Данные

Рандомизированный эксперимент: воздействие получает случайная пятая часть объектов, вероятность $0{,}2$ не зависит от признаков. Признаков пять, $X \sim \mathcal N(0, I)$.

- Эффект зависит только от первого признака: $\tau(x) = 1 + 1{,}5\,x_0$. У объектов с $x_0 < -2/3$ эффект отрицательный: воздействие им вредит.
- Исход без воздействия зависит от второго и третьего признаков: $\mu_0(x) = 2x_1 + x_2^2$. Эти признаки объясняют исход, но эффект не различают — как возраст в примере из урока «Причинный лес и честное расщепление».
- Наблюдаемый исход: $Y = \mu_0(X) + T\,\tau(X) + \varepsilon$, $\varepsilon \sim \mathcal N(0, 1)$.

Половина выборки идет на обучение, половина — на отложенную проверку.

In [ ]:
rng = np.random.default_rng(0)
n, d = 4000, 5
X = rng.normal(size=(n, d))
tau = 1 + 1.5 * X[:, 0]                       # истинный CATE
T = rng.binomial(1, 0.2, size=n)              # воздействие: случайная пятая часть
mu0 = 2 * X[:, 1] + X[:, 2] ** 2              # исход без воздействия
Y = mu0 + T * tau + rng.normal(size=n)

idx = rng.permutation(n)
tr, te = idx[: n // 2], idx[n // 2 :]
print('обучение:', len(tr), '|', 'проверка:', len(te), '|', 'доля воздействия:', round(T.mean(), 3))
print('доля объектов с отрицательным эффектом:', round((tau < 0).mean(), 3))

**Задание 1. Обучите S-, T- и X-learner из `econml.metalearners` на обучающей половине. Внутри всех схем — `GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=0)`, у X-learner модель склонности — `LogisticRegression()`. Посчитайте оценки CATE на проверочной половине (`tau_s`, `tau_t`, `tau_x`) и RMSE каждой относительно истинного `tau[te]`. Какая схема ближе всего к истинному эффекту?**

Оформите RMSE функцией `rmse(est)` и модель внутри схем — функцией `gb()`: они понадобятся дальше.

In [ ]:
# Ваш код здесь

Рассеяние «истинный эффект против оцененного»: по оси X истинный $\tau(x)$, по оси Y оценка схемы, красная линия — $y = x$. Чем ближе облако точек к линии, тем точнее схема восстанавливает эффект на отдельных объектах. Такую картинку можно нарисовать только на синтетических данных.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharex=True, sharey=True)
for ax, (name, est) in zip(axes, (('S-learner', tau_s), ('T-learner', tau_t), ('X-learner', tau_x))):
    ax.scatter(tau[te], est, s=6, alpha=0.35, color='#3b6ea5')
    lim = [tau[te].min(), tau[te].max()]
    ax.plot(lim, lim, color='#c0392b', lw=1.5, label='y = x')
    ax.set_title(f'{name}: RMSE = {rmse(est):.2f}')
    ax.set_xlabel('True effect τ(x)')
    ax.legend(loc='upper left')
axes[0].set_ylabel('Estimated effect τ̂(x)')
fig.suptitle('True versus estimated CATE on the held-out half')
fig.tight_layout()
fig.savefig('cate_true_vs_estimated.png', dpi=150, bbox_inches='tight')
plt.show()

**Задание 2. Обучите причинный лес `CausalForestDML` на обучающей половине: `model_y=gb()`, `model_t=GradientBoostingClassifier(random_state=0)`, `discrete_treatment=True`, `n_estimators=1000`, `min_samples_leaf=10`, `random_state=0`; честное расщепление в нем включено по умолчанию (`honest=True`). Посчитайте на проверочной половине оценку `tau_cf`, ее RMSE и 95-процентные доверительные интервалы `effect_interval(..., alpha=0.05)`. Какая доля объектов проверочной половины имеет истинный эффект внутри своего интервала?**

In [ ]:
# Ваш код здесь

### Калибровка по квантилям предсказанного эффекта

Дальше — проверки, которые не требуют знать истинный эффект. Отсортируем объекты проверочной половины по предсказанному эффекту и разобьем на четыре группы по четверти. Внутри каждой группы посчитаем фактическую разность средних исходов между воздействованными и контрольными: эксперимент рандомизирован, поэтому эта разность оценивает средний эффект в группе. Если модель ранжирует эффект осмысленно, фактическая разность растет вместе с предсказанной.

**Задание 3. Разбейте проверочную половину на квартили по `tau_cf` (`pd.qcut(tau_cf, 4, labels=[1, 2, 3, 4])`). Для каждого квартиля выведите средний предсказанный эффект, фактическую разность средних исходов воздействованных и контрольных и, для сравнения, средний истинный эффект. Растет ли фактическая разность от первого квартиля к четвертому?**

In [ ]:
# Ваш код здесь

### Uplift-кривая и кривая Qini

Калибровка по квартилям — четыре точки. Uplift-кривая делает то же для любой доли объектов: сортируем объекты по $\hat\tau(x)$ по убыванию, берем верхнюю долю $\varphi$ и считаем, какой прирост исхода на один объект всей выборки дало бы воздействие только на эту долю: $(\bar Y_{T=1}(\varphi) - \bar Y_{T=0}(\varphi))\cdot\varphi$. Случайный порядок дает прямую от нуля до разности средних исходов воздействованных и контрольных по всей выборке.

Кривая Qini устроена так же, но считает прирост через суммы: $Q(\varphi) = \bigl(S_{T=1}(\varphi) - S_{T=0}(\varphi)\cdot N_{T=1}(\varphi)/N_{T=0}(\varphi)\bigr)/N_{T=1}$, где $S$ — сумма исходов, $N$ — число объектов группы среди верхней доли $\varphi$. **Коэффициент Qini** — площадь между кривой Qini и прямой случайного порядка: чем она больше, тем лучше модель отделяет объекты с большим эффектом от объектов с малым.

**Задание 4. Напишите функцию `qini_curve(tau_hat, y, t)`, которая возвращает долю $\varphi$ (от 0 до 1 с шагом в один объект), значения $Q(\varphi)$ и коэффициент Qini — площадь $\int_0^1 \bigl(Q(\varphi) - \varphi\,Q(1)\bigr)\,d\varphi$ по формуле трапеций (`np.trapezoid`). Посчитайте коэффициент на проверочной половине для причинного леса, для X-learner и для сортировки по истинному эффекту `tau[te]`.**

In [ ]:
# Ваш код здесь

Слева — uplift-кривые причинного леса, X-learner и сортировки по истинному эффекту, пунктир — случайный порядок. Справа — кривая Qini причинного леса; закрашенная площадь между ней и прямой случайного порядка и есть коэффициент Qini.

In [ ]:
def uplift_curve(tau_hat, y, t):
    order = np.argsort(-tau_hat)
    y, t = y[order], t[order]
    n_t, n_c = np.cumsum(t), np.cumsum(1 - t)
    mean_t = np.divide(np.cumsum(y * t), n_t, out=np.zeros(len(y)), where=n_t > 0)
    mean_c = np.divide(np.cumsum(y * (1 - t)), n_c, out=np.zeros(len(y)), where=n_c > 0)
    k = np.arange(1, len(y) + 1)
    gain = np.where((n_t > 0) & (n_c > 0), (mean_t - mean_c) * k / len(y), 0.0)
    return np.r_[0, k / len(y)], np.r_[0, gain]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.6))
for name, est, color in (('Causal forest', tau_cf, '#3b6ea5'), ('X-learner', tau_x, '#e67e22'),
                         ('Sorted by true effect', tau[te], '#7f8c8d')):
    phi, g = uplift_curve(est, y_te, t_te)
    ax1.plot(phi, g, color=color, lw=1.8, ls='--' if name.startswith('Sorted') else '-', label=name)
ax1.plot([0, 1], [0, g[-1]], color='black', lw=1, ls=':', label='Random order')
ax1.set_title('Uplift curve')
ax1.set_xlabel('Share of objects treated, sorted by predicted effect')
ax1.set_ylabel('Cumulative gain per object')
ax1.legend()

phi, q, coef = qini_curve(tau_cf, y_te, t_te)
ax2.plot(phi, q, color='#3b6ea5', lw=1.8, label='Causal forest')
ax2.plot([0, 1], [0, q[-1]], color='black', lw=1, ls=':', label='Random order')
ax2.fill_between(phi, phi * q[-1], q, color='#3b6ea5', alpha=0.2, label=f'Qini coefficient = {coef:.3f}')
ax2.set_title('Qini curve')
ax2.set_xlabel('Share of objects treated, sorted by predicted effect')
ax2.set_ylabel('Qini: incremental gain per treated object')
ax2.legend()
fig.tight_layout()
fig.savefig('uplift_qini_curves.png', dpi=150, bbox_inches='tight')
plt.show()

**Выводы**

- На синтетике видно то, что в работе с реальными данными недоступно: насколько каждая оценка $\hat\tau(x)$ отличается от истинного эффекта на отдельном объекте. Сравните RMSE схем с тем, что урок «Мета-обучатели: S, T и X» говорит про дисбаланс групп: воздействие здесь получила пятая часть выборки.
- Доверительные интервалы причинного леса — асимптотические. Сравните долю объектов, у которых истинный эффект попал в интервал, с номинальными 95%.
- Калибровка по квантилям, uplift-кривая и коэффициент Qini не требуют знать истинный эффект, но проверяют только ранжирование по группам, а не точность на объекте.
- Признаки $x_1$ и $x_2$ определяют исход, но не эффект: модель исхода считала бы их важными, а для решения «кому назначать воздействие» они ничего не дают.

Источники:
1. [Künzel et al., 2019. Metalearners for estimating heterogeneous treatment effects using machine learning](https://arxiv.org/abs/1706.03461)
2. [Wager & Athey, 2018. Estimation and inference of heterogeneous treatment effects using random forests](https://arxiv.org/abs/1510.04342)
3. [EconML](https://github.com/py-why/EconML)